# Practical: Text-to-Image Generation with a Pretrained Stable Diffusion Model

## Aim
Use a **pretrained Stable Diffusion model** through the Hugging Face **Diffusers** library to generate images from text prompts, and analyse how **(1) the number of inference steps, (2) the guidance scale (classifier-free guidance)** and **(3) the sampling method (scheduler)** affect image quality.

## Theory in brief
* **Latent diffusion.** Stable Diffusion runs the diffusion process in a compressed *latent* space. A **CLIP text encoder** turns the prompt into embeddings, a **U-Net** predicts the noise in the latent at every step, a **scheduler (sampler)** uses that prediction to take one denoising step, and a **VAE decoder** converts the final latent into pixels.
* **Inference steps.** More steps = finer discretisation of the reverse process = usually better detail, but linearly more compute.
* **Guidance scale (CFG).** At each step the U-Net is run with and without the prompt; the final noise prediction is `eps_uncond + w * (eps_cond - eps_uncond)`. Larger `w` follows the prompt more strongly but reduces diversity and can cause saturation/artefacts. `w <= 1` disables CFG (and is about 2x faster).
* **Scheduler / sampler.** Different numerical solvers (DDIM, Euler, DPM-Solver++, UniPC ...) trade speed against quality; modern solvers reach good quality in far fewer steps than the original PNDM/DDPM-style samplers.

## Plan
1. Install and load the model
2. Baseline generation
3. Helper functions and quantitative metrics (CLIP score, sharpness, convergence error, time)
4. Experiment 1: inference steps
5. Experiment 2: guidance scale
6. Experiment 3: sampling methods
7. Combined experiment: scheduler x steps
8. Analysis and conclusion

## 1. Setup

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate safetensors scipy pandas

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy import ndimage
from PIL import Image
warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
print("Device:", DEVICE, "| dtype:", DTYPE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU found. In Colab use Runtime > Change runtime type > T4 GPU. CPU will be very slow.")

## 2. Load the pretrained Stable Diffusion pipeline
`segmind/tiny-sd` is a distilled, lightweight Stable Diffusion model (as used in the reference notebook) and runs quickly on a free Colab GPU.
For higher quality you can switch to the full SD 1.5 weights by changing `MODEL_ID` (needs more VRAM and time per image).

In [ ]:
from diffusers import DiffusionPipeline

MODEL_ID = "segmind/tiny-sd"
# MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"   # optional: full model

pipe = DiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    safety_checker=None,            # avoids false-positive black images in the experiments
    requires_safety_checker=False,
).to(DEVICE)
pipe.set_progress_bar_config(disable=True)

BASE_SCHEDULER_CONFIG = pipe.scheduler.config
print("Default scheduler:", pipe.scheduler.__class__.__name__)
print("Components:", list(pipe.components.keys()))

### Pipeline components
Printing parameter counts makes the architecture described in the theory section concrete.

In [ ]:
def n_params(m): return sum(p.numel() for p in m.parameters()) / 1e6

for name in ["text_encoder", "unet", "vae"]:
    print(f"{name:<13}: {n_params(getattr(pipe, name)):8.1f} M parameters  ({getattr(pipe, name).__class__.__name__})")

## 3. Baseline generation

In [ ]:
PROMPT = (
    "A realistic smart agricultural farm, green crops, automated irrigation systems, "
    "drones monitoring the fields, clear blue sky, high quality"
)

generator = torch.Generator(device=DEVICE).manual_seed(42)
image = pipe(PROMPT, num_inference_steps=25, guidance_scale=7.5, generator=generator).images[0]

plt.figure(figsize=(6, 6))
plt.imshow(image); plt.axis("off")
plt.title("Baseline: 25 steps, guidance 7.5, default scheduler")
plt.show()

## 4. Helper functions and metrics

Fair comparison needs a **fixed seed** (same initial noise) so that only the parameter under study changes.
We also measure objective proxies for quality, because visual judgement alone is subjective:

| Metric | What it measures | Better when |
|---|---|---|
| **CLIP score** | cosine similarity between image and prompt embeddings (x100) | higher = better prompt alignment |
| **Sharpness** | variance of the Laplacian of the grayscale image | higher = more fine detail (very high can mean noise/artefacts) |
| **Saturation** | mean HSV saturation | extreme values reveal over-saturation at high guidance |
| **Distance to reference (MSE)** | pixel MSE against a high-quality reference (many steps) | lower = closer to the converged result |
| **Time (s)** | wall-clock time for one image | lower = faster |

These are proxies, not ground truth: always inspect the images as well.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

clip_id   = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_id).to(DEVICE).eval()
clip_proc  = CLIPProcessor.from_pretrained(clip_id)

@torch.no_grad()
def clip_score(img, text):
    inputs = clip_proc(text=[text], images=img, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    out = clip_model(**inputs)
    i = out.image_embeds / out.image_embeds.norm(dim=-1, keepdim=True)
    t = out.text_embeds  / out.text_embeds.norm(dim=-1, keepdim=True)
    return float((i * t).sum() * 100)

def sharpness(img):
    g = np.asarray(img.convert("L"), dtype=np.float32)
    return float(ndimage.laplace(g).var())

def saturation(img):
    return float(np.asarray(img.convert("HSV"), dtype=np.float32)[..., 1].mean() / 255.0)

def mse(a, b):
    a = np.asarray(a, dtype=np.float32) / 255.0
    b = np.asarray(b, dtype=np.float32) / 255.0
    return float(((a - b) ** 2).mean())

In [ ]:
from diffusers import (
    PNDMScheduler, DDIMScheduler, DDPMScheduler,
    EulerDiscreteScheduler, EulerAncestralDiscreteScheduler,
    HeunDiscreteScheduler, LMSDiscreteScheduler,
    DPMSolverMultistepScheduler, UniPCMultistepScheduler,
)

SCHEDULERS = {
    "PNDM (default SD1.5)": (PNDMScheduler, {}),
    "DDIM":                 (DDIMScheduler, {}),
    "Euler":                (EulerDiscreteScheduler, {}),
    "Euler Ancestral":      (EulerAncestralDiscreteScheduler, {}),
    "LMS":                  (LMSDiscreteScheduler, {}),
    "Heun (2nd order)":     (HeunDiscreteScheduler, {}),
    "DPM++ 2M":             (DPMSolverMultistepScheduler, {}),
    "DPM++ 2M Karras":      (DPMSolverMultistepScheduler, {"use_karras_sigmas": True}),
    "UniPC":                (UniPCMultistepScheduler, {}),
}

def set_scheduler(name):
    cls, kw = SCHEDULERS[name]
    pipe.scheduler = cls.from_config(BASE_SCHEDULER_CONFIG, **kw)

def sync():
    if DEVICE == "cuda": torch.cuda.synchronize()

def generate(prompt, steps=25, cfg=7.5, seed=42, scheduler="DPM++ 2M"):
    """Generate one image with a fixed seed. Returns (PIL image, seconds)."""
    set_scheduler(scheduler)
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    sync(); t0 = time.time()
    img = pipe(prompt, num_inference_steps=steps, guidance_scale=cfg, generator=g).images[0]
    sync(); dt = time.time() - t0
    return img, dt

def evaluate(img, prompt, dt, ref=None):
    row = {"CLIP": clip_score(img, prompt), "Sharpness": sharpness(img),
           "Saturation": saturation(img), "Time (s)": dt}
    if ref is not None: row["MSE vs ref"] = mse(img, ref)
    return row

def show_grid(images, titles, ncols=4, size=3.2, suptitle=None):
    nrows = int(np.ceil(len(images) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(size * ncols, (size + 0.4) * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes: ax.axis("off")
    for ax, im, t in zip(axes, images, titles):
        ax.imshow(im); ax.set_title(t, fontsize=10)
    if suptitle: fig.suptitle(suptitle, fontsize=13, y=1.0)
    plt.tight_layout(); plt.show()

SEEDS = [42, 7, 123]       # metrics are averaged over these seeds; grids show the first seed

In [ ]:
# Warm-up run so that the first timing measurement is not inflated by CUDA initialisation
_ = generate(PROMPT, steps=5)
print("Warm-up done.")

## 5. Experiment 1: Effect of the number of inference steps
**Fixed:** prompt, seed, guidance = 7.5, scheduler = DPM++ 2M. **Varied:** steps in {5, 10, 15, 20, 30, 50}.
A 100-step image is generated as the *converged reference*, and the MSE to it shows how quickly the result stabilises.

In [ ]:
STEP_LIST = [5, 10, 15, 20, 30, 50]

# reference: many steps, per seed
refs = {s: generate(PROMPT, steps=100, cfg=7.5, seed=s, scheduler="DPM++ 2M")[0] for s in SEEDS}

rows, grid_imgs = [], {}
for steps in STEP_LIST:
    for s in SEEDS:
        img, dt = generate(PROMPT, steps=steps, cfg=7.5, seed=s, scheduler="DPM++ 2M")
        rows.append({"Steps": steps, "Seed": s, **evaluate(img, PROMPT, dt, refs[s])})
        if s == SEEDS[0]: grid_imgs[steps] = img

df_steps = pd.DataFrame(rows)
show_grid([grid_imgs[k] for k in STEP_LIST], [f"{k} steps" for k in STEP_LIST], ncols=3,
          suptitle="Experiment 1: inference steps (same seed, guidance 7.5)")
summary_steps = df_steps.groupby("Steps").mean(numeric_only=True).drop(columns="Seed").round(4)
summary_steps

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(18, 3.8))
for a, col in zip(ax, ["CLIP", "Sharpness", "MSE vs ref", "Time (s)"]):
    a.plot(summary_steps.index, summary_steps[col], marker="o"); a.set_title(col)
    a.set_xlabel("Inference steps"); a.grid(alpha=.3)
plt.suptitle("Experiment 1: metrics vs number of inference steps (mean of 3 seeds)", y=1.03)
plt.tight_layout(); plt.show()

### Observations: Experiment 1 (inference steps)
*Typical behaviour (confirm against your own grid and plots above):*
* **Very low steps (5 to 10):** images are blurry, blobby or under-denoised; structures such as crops, drones and irrigation lines are vague. The MSE to the reference is highest here.
* **Moderate steps (15 to 25):** the scene becomes coherent and details appear. Most of the quality gain happens in this range.
* **High steps (30 to 50+):** differences become small. MSE to the reference flattens out and CLIP score plateaus, so extra steps mostly cost time.
* **Time grows linearly** with the number of steps (each step is one U-Net forward pass, two with CFG).
* **Conclusion:** there are diminishing returns. About 20 to 30 steps is a good quality/speed trade-off for modern solvers.

## 6. Experiment 2: Effect of the guidance scale
**Fixed:** prompt, seed, 25 steps, scheduler = DPM++ 2M. **Varied:** guidance in {1, 3, 5, 7.5, 10, 15, 20}.

In [ ]:
CFG_LIST = [1, 3, 5, 7.5, 10, 15, 20]

rows, grid_imgs = [], {}
for cfg in CFG_LIST:
    for s in SEEDS:
        img, dt = generate(PROMPT, steps=25, cfg=cfg, seed=s, scheduler="DPM++ 2M")
        rows.append({"Guidance": cfg, "Seed": s, **evaluate(img, PROMPT, dt)})
        if s == SEEDS[0]: grid_imgs[cfg] = img

df_cfg = pd.DataFrame(rows)
show_grid([grid_imgs[k] for k in CFG_LIST], [f"guidance = {k}" for k in CFG_LIST], ncols=4,
          suptitle="Experiment 2: guidance scale (same seed, 25 steps)")
summary_cfg = df_cfg.groupby("Guidance").mean(numeric_only=True).drop(columns="Seed").round(4)
summary_cfg

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(18, 3.8))
for a, col in zip(ax, ["CLIP", "Sharpness", "Saturation", "Time (s)"]):
    a.plot(summary_cfg.index, summary_cfg[col], marker="o", color="tab:green"); a.set_title(col)
    a.set_xlabel("Guidance scale"); a.grid(alpha=.3)
plt.suptitle("Experiment 2: metrics vs guidance scale (mean of 3 seeds)", y=1.03)
plt.tight_layout(); plt.show()

In [ ]:
# Diversity: does a higher guidance scale reduce variety across seeds?
div_seeds = [1, 2, 3, 4]
rows_imgs, rows_titles = [], []
for cfg in [3, 7.5, 15]:
    for s in div_seeds:
        rows_imgs.append(generate(PROMPT, steps=25, cfg=cfg, seed=s)[0]); rows_titles.append(f"cfg {cfg}, seed {s}")
show_grid(rows_imgs, rows_titles, ncols=4, size=2.8, suptitle="Diversity across seeds at low / medium / high guidance")

### Observations: Experiment 2 (guidance scale)
*Typical behaviour (confirm against your own outputs):*
* **guidance about 1:** the prompt is largely ignored; images are vague, low-contrast and often incoherent. (CFG is switched off, so it is also the fastest setting.)
* **guidance 3 to 5:** natural-looking, varied images that only loosely follow the prompt.
* **guidance 7 to 10:** the best balance. Prompt elements (crops, drones, blue sky) are clearly present and CLIP score is usually at or near its peak.
* **guidance 15 to 20:** over-saturated colours, harsh contrast, burnt textures and artefacts; sharpness may rise but because of noise rather than real detail. Diversity across seeds drops, so images look more alike.
* **Conclusion:** guidance trades **fidelity to the prompt** against **naturalness and diversity**. Around 7 to 8 is a sensible default for SD 1.5-type models.

## 7. Experiment 3: Effect of the sampling method (scheduler)
Compare nine schedulers on the same prompt and seed.
* **3a:** all samplers at 25 steps (does the final quality differ?).
* **3b:** all samplers at only 10 steps (which converge fastest?).

In [ ]:
def scheduler_experiment(steps, label):
    ref_cache = refs
    rows, grid = [], {}
    for name in SCHEDULERS:
        for s in SEEDS:
            img, dt = generate(PROMPT, steps=steps, cfg=7.5, seed=s, scheduler=name)
            rows.append({"Scheduler": name, "Seed": s, **evaluate(img, PROMPT, dt, ref_cache[s])})
            if s == SEEDS[0]: grid[name] = img
    df = pd.DataFrame(rows)
    show_grid([grid[n] for n in SCHEDULERS], list(SCHEDULERS), ncols=3, size=3.4,
              suptitle=f"Experiment 3{label}: schedulers at {steps} steps (same seed, guidance 7.5)")
    return df, df.groupby("Scheduler").mean(numeric_only=True).drop(columns="Seed").round(4).loc[list(SCHEDULERS)]

df_s25, summary_s25 = scheduler_experiment(25, "a")
summary_s25

In [ ]:
df_s10, summary_s10 = scheduler_experiment(10, "b")
summary_s10

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.6))
x = np.arange(len(SCHEDULERS)); w = 0.4
for a, col in zip(ax, ["CLIP", "MSE vs ref", "Time (s)"]):
    a.bar(x - w/2, summary_s10[col], w, label="10 steps")
    a.bar(x + w/2, summary_s25[col], w, label="25 steps")
    a.set_xticks(x); a.set_xticklabels(list(SCHEDULERS), rotation=45, ha="right"); a.set_title(col); a.legend()
plt.suptitle("Experiment 3: scheduler comparison (mean of 3 seeds)", y=1.02)
plt.tight_layout(); plt.show()

### Observations: Experiment 3 (sampling methods)
*Typical behaviour (confirm against your own outputs):*
* **Convergence speed at 10 steps:** higher-order solvers (**DPM++ 2M, DPM++ 2M Karras, UniPC**) already produce clean, detailed images, while **PNDM, DDIM, LMS** are noticeably softer or noisier. Euler is in between.
* **At 25 steps** most samplers look similar in quality; differences become matters of style, texture and composition rather than correctness.
* **Ancestral samplers (Euler a)** inject fresh noise at every step, so they do *not* converge to a single image: results change with the step count and are more varied/creative but less reproducible.
* **Deterministic samplers (DDIM, Euler, DPM++, UniPC)** converge to a stable image as steps increase.
* **Heun** is second order, so it evaluates the U-Net twice per step: good quality but roughly 2x slower for the same step count.
* **Karras sigmas** spend more of the step budget at low noise levels, which often sharpens fine detail.
* **Conclusion:** for a fixed compute budget, **DPM++ 2M (Karras) or UniPC** usually give the best quality per second; DDIM/PNDM need more steps for comparable results.

## 8. Combined experiment: scheduler x steps
How do a few representative samplers behave as the step budget grows? The grid shows rows = scheduler, columns = steps.

In [ ]:
combo_sched = ["PNDM (default SD1.5)", "DDIM", "Euler Ancestral", "DPM++ 2M Karras"]
combo_steps = [5, 10, 20, 40]
imgs, titles = [], []
for sc in combo_sched:
    for st in combo_steps:
        imgs.append(generate(PROMPT, steps=st, cfg=7.5, seed=SEEDS[0], scheduler=sc)[0]); titles.append(f"{sc}\n{st} steps")
show_grid(imgs, titles, ncols=len(combo_steps), size=3.0, suptitle="Scheduler x steps")

## 9. Generalisation check: other prompts
The conclusions should not depend on one prompt. Here we repeat a compact sweep with different prompt styles: the default scheduler at 25 steps while changing guidance, and then the best scheduler at low steps.

In [ ]:
OTHER_PROMPTS = {
    "portrait": "Portrait photo of an elderly fisherman, detailed face, soft natural light, 85mm lens",
    "landscape": "A snowy mountain village at sunrise, warm light, highly detailed, cinematic",
    "object": "A red vintage bicycle leaning on a brick wall, studio lighting, sharp focus",
}
rows = []
imgs, titles = [], []
for pname, p in OTHER_PROMPTS.items():
    for cfg in [2, 7.5, 18]:
        img, dt = generate(p, steps=25, cfg=cfg, seed=42, scheduler="DPM++ 2M")
        rows.append({"Prompt": pname, "Guidance": cfg, **evaluate(img, p, dt)})
        imgs.append(img); titles.append(f"{pname}, cfg={cfg}")
show_grid(imgs, titles, ncols=3, size=3.4, suptitle="Guidance effect on different prompts")
pd.DataFrame(rows).round(3)

## 10. Answers to the reference-notebook questions
**Does the generated image match the text prompt?** Mostly yes at guidance 7 to 10 with 20+ steps: the main elements (green crops, sky, field layout) appear. Small objects such as drones and irrigation hardware are the most likely to be missing or distorted. The CLIP score gives a numeric estimate.

**What happens when we change the prompt?** The subject, style and composition change, but the same parameter trends hold (see Section 9): low guidance ignores the prompt, high guidance over-saturates. More specific prompts (subject + style + lighting + quality cues) give more controllable results.

**Are the objects realistic?** Textures such as foliage, sky and terrain are usually convincing; man-made or small objects (drones, machinery) and faces/hands are the weak spots, especially for the small `tiny-sd` model and at low step counts.

**Does the model generate unwanted or incorrect details?** Yes, occasionally: duplicated or merged objects, malformed drones, garbled text, unnatural symmetry, and over-saturation at high guidance. Mitigations: more steps, moderate guidance, a better solver, negative prompts, and a larger model.

## 11. Overall analysis and conclusion

| Parameter | Too low | Sweet spot | Too high |
|---|---|---|---|
| **Inference steps** | blurry, incomplete denoising | about 20 to 30 | negligible gain, wasted compute |
| **Guidance scale** | ignores prompt, washed out, incoherent | about 7 to 10 | over-saturated, artefacts, low diversity |
| **Sampler** | PNDM / DDIM need more steps for equal quality | DPM++ 2M (Karras) / UniPC give good images in about 10 to 20 steps | Heun costs 2x per step; ancestral samplers never fully converge |

**Key takeaways**
1. Image quality is controlled jointly by *steps*, *guidance* and *sampler*, and these interact: a better sampler lets you use fewer steps.
2. Guidance is a **fidelity vs. naturalness/diversity** knob, not a "quality" knob.
3. Steps show **diminishing returns**; compute grows linearly while quality saturates.
4. Always fix the **seed** when comparing, and combine visual inspection with metrics such as CLIP score, since each metric alone can mislead (e.g. sharpness rises with noise).

**Limitations:** metrics are proxies; only a few seeds/prompts were used; `tiny-sd` is a distilled model, so absolute quality is below full SD 1.5/SDXL, though the trends are the same.

## 12. Save results

In [ ]:
import os
os.makedirs("results", exist_ok=True)
df_steps.to_csv("results/exp1_steps.csv", index=False)
df_cfg.to_csv("results/exp2_guidance.csv", index=False)
df_s25.to_csv("results/exp3_schedulers_25steps.csv", index=False)
df_s10.to_csv("results/exp3_schedulers_10steps.csv", index=False)
image.save("results/baseline.png")
print("Saved CSVs and baseline image to ./results")